In [ ]:
import scanpy as sc
import anndata as ad
import pandas as pd
import warnings; warnings.simplefilter('ignore')
import os
import numpy as np
import matplotlib.pyplot as plt
import scipy.sparse as sparse
from scipy.stats import wilcoxon
from statsmodels.stats.multitest import multipletests

In [ ]:
import sys
from pathlib import Path

_p = Path.cwd().resolve()
while not (_p / 'config.yaml').exists() and _p != _p.parent:
    _p = _p.parent
sys.path.insert(0, str(_p / 'scripts'))
from paths import P, ensure_dirs

In [ ]:
plt.rcParams.update({
    "font.family": "Arial",
    "font.size": 12,
})
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42

# Data loading

In [ ]:
ADATA_PATH = str(P.processed.adata.all_cells / P.fn.all_cells_final)

all_cells_adata = sc.read_h5ad(ADATA_PATH)
epi_adata = sc.read_h5ad(str(P.processed.adata.epithelial / "epithelial_20_50_harmony_batch_05_pt_05_ssg.h5ad"))
all_cells_adata = all_cells_adata[~all_cells_adata.obs['cell_id'].duplicated(keep=False)].copy()

output_dir = str(P.results.figures / "supp_9")
os.makedirs(output_dir, exist_ok=True)

# GDF15 comparison boxplots

In [ ]:
def pval_to_stars(p):
    if p < 0.001: return "***"
    elif p < 0.01: return "**"
    elif p < 0.05: return "*"
    else: return "ns"

gene = "GDF15"
adata = all_cells_adata

gene_idx = adata.var_names.get_loc(gene)
X = adata.X
if sparse.issparse(X):
    expr = np.array(X[:, gene_idx].todense()).flatten()
else:
    expr = X[:, gene_idx]

df_all = pd.DataFrame({
    "GDF15": expr,
    "core_id": adata.obs["core_id"].values,
    "patient_id": adata.obs["patient_id"].values,
    "cell_type_coarse": adata.obs["annotation_final_coarse"].values,
    "cell_type_fine": adata.obs["annotation_final_fine"].values,
})

fib_mask = df_all["cell_type_fine"].str.contains("Fibroblast|CAF", case=False, na=False)
df_all.loc[fib_mask, "cell_type_fine"] = "Fibroblasts"

all_cells_adata_uniform = all_cells_adata[all_cells_adata.obs['mixed_core_tissue_type'] == False].copy()
epi_mask_u = all_cells_adata_uniform.obs['annotation_final_coarse'] == 'Epithelial'
epi_obs_u = all_cells_adata_uniform.obs[epi_mask_u]
core_tissue_map = epi_obs_u.groupby('core_id')['tissue_type_cell_level_normal_split'].first()

print("Cores with mixed epithelial tissue types:")
mixed = epi_obs_u.groupby('core_id')['tissue_type_cell_level_normal_split'].nunique()
print(mixed[mixed > 1])

all_cells_adata_uniform.obs['core_tissue_type'] = all_cells_adata_uniform.obs['core_id'].map(core_tissue_map)
print("\ncore_tissue_type value counts:")
print(all_cells_adata_uniform.obs['core_tissue_type'].value_counts())

adata_u = all_cells_adata_uniform
gene_idx_u = adata_u.var_names.get_loc(gene)
X_u = adata_u.X
if sparse.issparse(X_u):
    expr_u = np.array(X_u[:, gene_idx_u].todense()).flatten()
else:
    expr_u = X_u[:, gene_idx_u]

In [ ]:
# GDF15 Epithelial vs Non-epithelial (all patients pooled)
df1 = df_all.copy()
df1["is_epithelial"] = df1["cell_type_coarse"] == "Epithelial"

core_df = (
    df1.groupby(["patient_id", "core_id", "is_epithelial"])["GDF15"]
    .mean()
    .reset_index()
    .pivot_table(index=["patient_id", "core_id"], columns="is_epithelial", values="GDF15")
    .rename(columns={True: "epi_mean", False: "non_epi_mean"})
    .dropna()
    .reset_index()
)

patient_df = (
    core_df.groupby("patient_id")[["epi_mean", "non_epi_mean"]]
    .mean()
    .reset_index()
)

stat, pval1 = wilcoxon(patient_df["epi_mean"], patient_df["non_epi_mean"], alternative="two-sided")
print(f"\n=== Epithelial vs Non-epithelial ===")
print(f"  n={len(patient_df)}, p={pval1:.4e}\n")

fig, ax = plt.subplots(figsize=(3.5, 5))
data = [patient_df["epi_mean"].values, patient_df["non_epi_mean"].values]
colors = ["#34ace0", "#84817a"]
bp = ax.boxplot(data, positions=[0, 1], widths=0.45, patch_artist=True,
                medianprops=dict(color="black", linewidth=2),
                whiskerprops=dict(linewidth=1.2), capprops=dict(linewidth=1.2), showfliers=False)
for patch, color in zip(bp["boxes"], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.85)
for i, col in enumerate(["epi_mean", "non_epi_mean"]):
    jitter = np.random.uniform(-0.07, 0.07, size=len(patient_df))
    ax.scatter(i + jitter, patient_df[col], color="black", s=10, zorder=3, alpha=0.6)
y_max = max(patient_df[["epi_mean", "non_epi_mean"]].max()) * 1.12
bracket_h = y_max * 0.04
ax.plot([0, 0, 1, 1], [y_max, y_max + bracket_h, y_max + bracket_h, y_max], color="black", linewidth=1.2)
ax.text(0.5, y_max + bracket_h * 1.3, f"{pval_to_stars(pval1)}\np={pval1:.3f}", ha="center", va="bottom", fontsize=10)
ax.set_xticks([0, 1])
ax.set_xticklabels(["Epithelial", "Non-epithelial"])
ax.set_ylabel("GDF15 expression\n(mean per patient)")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
fig.savefig(os.path.join(output_dir, "fig1_GDF15_epi_vs_nonepi.pdf"), dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# GDF15 by coarse cell type (paired Wilcoxon vs Epithelial)
patient_means = (
    df_all.groupby(["patient_id", "core_id", "cell_type_coarse"])["GDF15"]
    .mean()
    .reset_index()
    .groupby(["patient_id", "cell_type_coarse"])["GDF15"]
    .mean()
    .reset_index()
)

all_cell_types = sorted(patient_means["cell_type_coarse"].unique())
patients_per_type = patient_means.groupby("cell_type_coarse")["patient_id"].apply(set)
shared_patients = set.intersection(*patients_per_type.values)
patient_means_f = patient_means[patient_means["patient_id"].isin(shared_patients)]

epi_vals_2 = patient_means_f[patient_means_f["cell_type_coarse"] == "Epithelial"].sort_values("patient_id")["GDF15"].values
other_types = [ct for ct in all_cell_types if ct != "Epithelial"]

results2 = []
for ct in other_types:
    other_vals = patient_means_f[patient_means_f["cell_type_coarse"] == ct].sort_values("patient_id")["GDF15"].values
    stat, pval = wilcoxon(epi_vals_2, other_vals, alternative="two-sided")
    results2.append({"cell_type": ct, "n_paired": len(epi_vals_2), "other_vals": other_vals, "pval": pval})

results2_df = pd.DataFrame(results2)
_, padj, _, _ = multipletests(results2_df["pval"], method="fdr_bh")
results2_df["padj"] = padj

print(f"=== By coarse cell type ===")
print(results2_df[["cell_type", "n_paired", "pval", "padj"]].to_string(index=False), "\n")

desired_order = ["Epithelial", "Stromal", "Immune"]
results2_df["cell_type"] = pd.Categorical(results2_df["cell_type"],
    categories=[ct for ct in desired_order if ct != "Epithelial"], ordered=True)
results2_df = results2_df.sort_values("cell_type").reset_index(drop=True)

all_types = ["Epithelial"] + list(results2_df["cell_type"])
all_vals = [epi_vals_2] + [results2_df.loc[results2_df["cell_type"] == ct, "other_vals"].values[0]
            for ct in desired_order if ct != "Epithelial"]

cell_type_colors = {"Epithelial": "#34ace0", "Stromal": "#27ae60", "Immune": "#e55039"}

n = len(all_types)
fig, ax = plt.subplots(figsize=(max(5, n * 1.1), 5))
positions = np.arange(n)
colors = [cell_type_colors.get(ct, "#95a5a6") for ct in all_types]
bp = ax.boxplot(all_vals, positions=positions, widths=0.5, patch_artist=True,
                medianprops=dict(color="black", linewidth=2),
                whiskerprops=dict(linewidth=1.2), capprops=dict(linewidth=1.2), showfliers=False)
for patch, color in zip(bp["boxes"], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.85)
for i, vals in enumerate(all_vals):
    jitter = np.random.uniform(-0.07, 0.07, size=len(vals))
    ax.scatter(i + jitter, vals, color="black", s=10, zorder=3, alpha=0.6)

y_data_max = max([np.max(v) for v in all_vals])
y_base = y_data_max * 1.08
bracket_step = y_data_max * 0.15
for i, row in enumerate(results2_df.itertuples()):
    pos = i + 1
    y_max = y_base + i * bracket_step
    bracket_h = y_data_max * 0.03
    ax.plot([0, 0, pos, pos], [y_max, y_max + bracket_h, y_max + bracket_h, y_max], color="black", linewidth=1.0)
    ax.text((0 + pos) / 2, y_max + bracket_h * 1.3, f"{pval_to_stars(row.padj)}\np={row.padj:.3f}",
            ha="center", va="bottom", fontsize=8)

ax.set_xticks(positions)
ax.set_xticklabels(all_types, rotation=45, ha="right")
ax.set_ylabel("GDF15 expression\n(mean per patient)")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
fig.savefig(os.path.join(output_dir, "fig2_GDF15_by_celltype.pdf"), dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# GDF15 Epithelial vs Non-epithelial by tissue type
df3 = pd.DataFrame({
    "GDF15": expr_u,
    "core_id": adata_u.obs["core_id"].values,
    "patient_id": adata_u.obs["patient_id"].values,
    "core_tissue_type": adata_u.obs["core_tissue_type"].astype(str).values,
    "is_epithelial": (adata_u.obs["annotation_final_coarse"] == "Epithelial").values,
})
df3 = df3[df3["core_tissue_type"] != "NA"]

core_means3 = (
    df3.groupby(["patient_id", "core_id", "core_tissue_type", "is_epithelial"])["GDF15"]
    .mean()
    .reset_index()
)
patient_means3 = (
    core_means3.groupby(["patient_id", "core_tissue_type", "is_epithelial"])["GDF15"]
    .mean()
    .reset_index()
)

print(f"=== Epithelial vs Non-epithelial by tissue type ===")
print(f"Tissue types found: {sorted(patient_means3['core_tissue_type'].unique())}")

results3 = []
for tt in sorted(patient_means3["core_tissue_type"].unique()):
    tt_df = patient_means3[patient_means3["core_tissue_type"] == tt].dropna(subset=["GDF15"])
    tt_epi = tt_df[tt_df["is_epithelial"] == True].sort_values("patient_id")
    tt_non = tt_df[tt_df["is_epithelial"] == False].sort_values("patient_id")
    shared = set(tt_epi["patient_id"]) & set(tt_non["patient_id"])
    tt_epi = tt_epi[tt_epi["patient_id"].isin(shared)].sort_values("patient_id")
    tt_non = tt_non[tt_non["patient_id"].isin(shared)].sort_values("patient_id")
    epi_v = tt_epi["GDF15"].values
    non_v = tt_non["GDF15"].values
    print(f"  {tt}: n_paired={len(epi_v)}")
    if len(epi_v) < 3:
        print(f"    Skipping: too few paired observations")
        continue
    stat, pval = wilcoxon(epi_v, non_v, alternative="two-sided")
    results3.append({"tissue_type": tt, "n_paired": len(epi_v), "epi_mean": epi_v.mean(),
                     "non_epi_mean": non_v.mean(), "pval": pval,
                     "epi_vals": epi_v, "non_epi_vals": non_v})

results3_df = pd.DataFrame(results3)
_, padj, _, _ = multipletests(results3_df["pval"], method="fdr_bh")
results3_df["padj"] = padj
print(results3_df[["tissue_type", "n_paired", "epi_mean", "non_epi_mean", "pval", "padj"]].to_string(index=False), "\n")

tissue_order3 = [tt for tt in ["Dist_N", "Adj_N", "AD", "CA"] if tt in results3_df["tissue_type"].values]
results3_df["tissue_type"] = pd.Categorical(results3_df["tissue_type"], categories=tissue_order3, ordered=True)
results3_df = results3_df.sort_values("tissue_type")

tissue_colors3 = {
    "Dist_N": ["#a29bfe", "#6c5ce7"],
    "Adj_N":  ["#706fd3", "#474787"],
    "AD": ["#fcd092", "#ffb142"],
    "CA": ["#de6868", "#b33939"],
}

bracket_raise3 = {
    "Dist_N": 1.8,
    "Adj_N":  1.5,
    "AD": 1.1,
    "CA": 1.5,
}

n_panels = len(results3_df)
fig, axes = plt.subplots(1, n_panels, figsize=(3.2 * n_panels, 5), sharey=True)
if n_panels == 1:
    axes = [axes]

for ax, row in zip(axes, results3_df.itertuples()):
    colors = tissue_colors3.get(row.tissue_type, ["#95a5a6", "#7f8c8d"])
    bp = ax.boxplot([row.epi_vals, row.non_epi_vals], positions=[0, 1], widths=0.45, patch_artist=True,
                    medianprops=dict(color="black", linewidth=2),
                    whiskerprops=dict(linewidth=1.2), capprops=dict(linewidth=1.2), showfliers=False)
    for patch, color in zip(bp["boxes"], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.85)
    for i, vals in enumerate([row.epi_vals, row.non_epi_vals]):
        jitter = np.random.uniform(-0.07, 0.07, size=len(vals))
        ax.scatter(i + jitter, vals, color="black", s=10, zorder=3, alpha=0.6)
    y_max = max(np.max(row.epi_vals), np.max(row.non_epi_vals)) * bracket_raise3.get(row.tissue_type, 1.25)
    bracket_h = y_max * 0.04
    ax.plot([0, 0, 1, 1], [y_max, y_max + bracket_h, y_max + bracket_h, y_max], color="black", linewidth=1.2)
    ax.text(0.5, y_max + bracket_h * 1.3, f"{pval_to_stars(row.padj)}\np={row.padj:.3f}",
            ha="center", va="bottom", fontsize=9)
    ax.set_xticks([0, 1])
    ax.set_xticklabels(["Epithelial", "Non-epithelial"], rotation=20, ha="right")
    ax.set_title(f"{row.tissue_type}\n", fontsize=14)
    ax.spines[["top", "right"]].set_visible(False)
    if ax == axes[0]:
        ax.set_ylabel("GDF15 expression\n(mean per patient)")

plt.tight_layout()
fig.savefig(os.path.join(output_dir, "fig3_GDF15_epi_vs_nonepi_by_tissue.pdf"), dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# GDF15 Epithelial vs Fibroblasts (pooled)
df4 = df_all[df_all["cell_type_fine"].isin(["Epithelial", "Fibroblasts"])].copy()

core_means4 = df4.groupby(["patient_id", "core_id", "cell_type_fine"])["GDF15"].mean().reset_index()
patient_means4 = core_means4.groupby(["patient_id", "cell_type_fine"])["GDF15"].mean().reset_index()

epi4 = patient_means4[patient_means4["cell_type_fine"] == "Epithelial"].sort_values("patient_id")
fib4 = patient_means4[patient_means4["cell_type_fine"] == "Fibroblasts"].sort_values("patient_id")
shared4 = set(epi4["patient_id"]) & set(fib4["patient_id"])
epi4 = epi4[epi4["patient_id"].isin(shared4)].sort_values("patient_id")
fib4 = fib4[fib4["patient_id"].isin(shared4)].sort_values("patient_id")
epi_v4 = epi4["GDF15"].values
fib_v4 = fib4["GDF15"].values

stat, pval4 = wilcoxon(epi_v4, fib_v4, alternative="two-sided")
print(f"=== Epithelial vs Fibroblasts ===")
print(f"  n={len(epi_v4)}, p={pval4:.4e}\n")

fig, ax = plt.subplots(figsize=(3.5, 5))
colors = ["#34ace0", "#ED4C67"]
bp = ax.boxplot([epi_v4, fib_v4], positions=[0, 1], widths=0.45, patch_artist=True,
                medianprops=dict(color="black", linewidth=2),
                whiskerprops=dict(linewidth=1.2), capprops=dict(linewidth=1.2), showfliers=False)
for patch, color in zip(bp["boxes"], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.85)
for i, vals in enumerate([epi_v4, fib_v4]):
    jitter = np.random.uniform(-0.07, 0.07, size=len(vals))
    ax.scatter(i + jitter, vals, color="black", s=10, zorder=3, alpha=0.6)
y_max = max(np.max(epi_v4), np.max(fib_v4)) * 1.12
bracket_h = y_max * 0.04
ax.plot([0, 0, 1, 1], [y_max, y_max + bracket_h, y_max + bracket_h, y_max], color="black", linewidth=1.2)
ax.text(0.5, y_max + bracket_h * 1.3, f"{pval_to_stars(pval4)}\np={pval4:.3f}", ha="center", va="bottom", fontsize=10)
ax.set_xticks([0, 1])
ax.set_xticklabels(["Epithelial", "Fibroblasts"])
ax.set_ylabel("GDF15 expression\n(mean per patient)")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
fig.savefig(os.path.join(output_dir, "fig4_GDF15_epi_vs_fibroblasts.pdf"), dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# GDF15 Epithelial vs Fibroblasts by tissue type
df5 = pd.DataFrame({
    "GDF15": expr_u,
    "core_id": adata_u.obs["core_id"].values,
    "patient_id": adata_u.obs["patient_id"].values,
    "core_tissue_type": adata_u.obs["core_tissue_type"].astype(str).values,
    "cell_type_fine": adata_u.obs["annotation_final_fine"].values,
    "cell_type_coarse": adata_u.obs["annotation_final_coarse"].values,
})

df5 = df5[df5["core_tissue_type"] != "NA"]

fib_mask5 = df5["cell_type_fine"].str.contains("Fibroblast|CAF", case=False, na=False)
df5["group"] = None
df5.loc[df5["cell_type_coarse"] == "Epithelial", "group"] = "Epithelial"
df5.loc[fib_mask5, "group"] = "Fibroblasts"
df5 = df5.dropna(subset=["group"])

core_means5 = df5.groupby(["patient_id", "core_id", "core_tissue_type", "group"])["GDF15"].mean().reset_index()
patient_means5 = core_means5.groupby(["patient_id", "core_tissue_type", "group"])["GDF15"].mean().reset_index()

print(f"=== Epithelial vs Fibroblasts by tissue type ===")

results5 = []
for tt in sorted(patient_means5["core_tissue_type"].dropna().unique()):
    tt_df = patient_means5[patient_means5["core_tissue_type"] == tt].dropna(subset=["GDF15"])
    epi = tt_df[tt_df["group"] == "Epithelial"].sort_values("patient_id")
    fib = tt_df[tt_df["group"] == "Fibroblasts"].sort_values("patient_id")
    shared = set(epi["patient_id"]) & set(fib["patient_id"])
    epi = epi[epi["patient_id"].isin(shared)].sort_values("patient_id")
    fib = fib[fib["patient_id"].isin(shared)].sort_values("patient_id")
    epi_v = epi["GDF15"].values
    fib_v = fib["GDF15"].values
    print(f"  {tt}: n_paired={len(epi_v)}")
    if len(epi_v) < 3:
        print(f"    Skipping: too few paired observations")
        continue
    stat, pval = wilcoxon(epi_v, fib_v, alternative="two-sided")
    results5.append({"tissue_type": tt, "n_paired": len(epi_v), "epi_mean": epi_v.mean(),
                     "fib_mean": fib_v.mean(), "pval": pval,
                     "epi_vals": epi_v, "fib_vals": fib_v})

results5_df = pd.DataFrame(results5)
_, padj, _, _ = multipletests(results5_df["pval"], method="fdr_bh")
results5_df["padj"] = padj
print(results5_df[["tissue_type", "n_paired", "epi_mean", "fib_mean", "pval", "padj"]].to_string(index=False), "\n")

display_order5 = [tt for tt in ["Dist_N", "Adj_N", "AD", "CA"] if tt in results5_df["tissue_type"].values]
results5_df["tissue_type"] = pd.Categorical(results5_df["tissue_type"], categories=display_order5, ordered=True)
results5_df = results5_df.sort_values("tissue_type")

box_colors = ["#34ace0", "#ED4C67"]

bracket_raise5 = {
    "Dist_N": 1.5,
    "Adj_N":  1.25,
    "AD": 1.12,
    "CA": 1.25,
}

n_panels = len(results5_df)
fig, axes = plt.subplots(1, n_panels, figsize=(3.2 * n_panels, 5), sharey=True)
if n_panels == 1:
    axes = [axes]

for ax, row in zip(axes, results5_df.itertuples()):
    bp = ax.boxplot([row.epi_vals, row.fib_vals], positions=[0, 1], widths=0.45, patch_artist=True,
                    medianprops=dict(color="black", linewidth=2),
                    whiskerprops=dict(linewidth=1.2), capprops=dict(linewidth=1.2), showfliers=False)
    for patch, color in zip(bp["boxes"], box_colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.85)
    for i, vals in enumerate([row.epi_vals, row.fib_vals]):
        jitter = np.random.uniform(-0.07, 0.07, size=len(vals))
        ax.scatter(i + jitter, vals, color="black", s=10, zorder=3, alpha=0.6)
    y_max = max(np.max(row.epi_vals), np.max(row.fib_vals)) * bracket_raise5.get(row.tissue_type, 1.25)
    bracket_h = y_max * 0.04
    ax.plot([0, 0, 1, 1], [y_max, y_max + bracket_h, y_max + bracket_h, y_max], color="black", linewidth=1.2)
    ax.text(0.5, y_max + bracket_h * 1.3, f"{pval_to_stars(row.padj)}\np={row.padj:.3f}",
            ha="center", va="bottom", fontsize=9)
    ax.set_xticks([0, 1])
    ax.set_xticklabels(["Epithelial", "Fibroblasts"], rotation=20, ha="right")
    ax.set_title(f"{row.tissue_type}\n", fontsize=14)
    ax.spines[["top", "right"]].set_visible(False)
    if ax == axes[0]:
        ax.set_ylabel("GDF15 expression\n(mean per patient)")

plt.tight_layout()
fig.savefig(os.path.join(output_dir, "fig5_GDF15_epi_vs_fibroblasts_by_tissue.pdf"), dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# GDF15 by tissue type (epithelial only)
gene = "GDF15"
gene_idx = epi_adata.var_names.get_loc(gene)
X = epi_adata.X
if sparse.issparse(X):
    expr_epi = np.array(X[:, gene_idx].todense()).flatten()
else:
    expr_epi = X[:, gene_idx]

df_epi_gdf15 = pd.DataFrame({
    "GDF15":      expr_epi,
    "core_id":    epi_adata.obs["core_id"].astype(str).values,
    "patient_id": epi_adata.obs["patient_id"].astype(str).values,
    "tissue_type": epi_adata.obs["tissue_type_cell_level_normal_split"].astype(str).values,
})

df_epi_gdf15 = df_epi_gdf15[~df_epi_gdf15["tissue_type"].isin(["nan", "NA", "None"])]

patient_means_epi = (
    df_epi_gdf15.groupby(["patient_id", "core_id", "tissue_type"])["GDF15"]
    .mean()
    .reset_index()
    .groupby(["patient_id", "tissue_type"])["GDF15"]
    .mean()
    .reset_index()
)

tissue_order = ["Dist_N", "Adj_N", "AD", "CA"]
pairs = [("Dist_N", "Adj_N"), ("Dist_N", "AD"), ("Dist_N", "CA"),
         ("Adj_N", "AD"), ("Adj_N", "CA"), ("AD", "CA")]

results_epi = []
for a, b in pairs:
    pair_pivot = (
        patient_means_epi.pivot_table(index="patient_id", columns="tissue_type", values="GDF15")
        [[a, b]].dropna()
    )
    print(f"{a} vs {b}: n={len(pair_pivot)}")
    if len(pair_pivot) < 3:
        print(f"  Skipping: too few paired observations")
        continue
    stat, pval = wilcoxon(pair_pivot[a].values, pair_pivot[b].values, alternative="two-sided")
    results_epi.append({"comparison": f"{a} vs {b}", "a": a, "b": b,
                    "n": len(pair_pivot),
                    "mean_a": pair_pivot[a].mean(), "mean_b": pair_pivot[b].mean(),
                    "pval": pval,
                    "vals_a": pair_pivot[a].values,
                    "vals_b": pair_pivot[b].values})

results_epi_df = pd.DataFrame(results_epi)
_, padj, _, _ = multipletests(results_epi_df["pval"], method="fdr_bh")
results_epi_df["padj"] = padj
print(results_epi_df[["comparison", "n", "mean_a", "mean_b", "pval", "padj"]].to_string(index=False))

tissue_colors_epi_gdf15 = {
    "Dist_N": "#33fd55",
    "Adj_N":  "#40407a",
    "AD": "#ffb142",
    "CA": "#b33939",
}

all_pivot = patient_means_epi.pivot_table(index="patient_id", columns="tissue_type", values="GDF15")
all_vals  = [all_pivot[tt].dropna().values for tt in tissue_order]
positions = np.arange(len(tissue_order))
colors    = [tissue_colors_epi_gdf15[tt] for tt in tissue_order]

fig, ax = plt.subplots(figsize=(6, 5))

bp = ax.boxplot(
    all_vals,
    positions=positions,
    widths=0.5,
    patch_artist=True,
    medianprops=dict(color="black", linewidth=2),
    whiskerprops=dict(linewidth=1.2),
    capprops=dict(linewidth=1.2),
    showfliers=False,
)
for patch, color in zip(bp["boxes"], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.85)

for i, vals in enumerate(all_vals):
    jitter = np.random.uniform(-0.07, 0.07, size=len(vals))
    ax.scatter(i + jitter, vals, color="black", s=10, zorder=3, alpha=0.6)

y_data_max = max([np.max(v) for v in all_vals])
y_base      = y_data_max * 1.08
foot_length = y_data_max * 0.02
bracket_step = y_data_max * 0.12

sig_idx = 0
for row in results_epi_df.itertuples():
    if row.padj >= 0.05:
        continue
    pos_a = tissue_order.index(row.a)
    pos_b = tissue_order.index(row.b)
    y_top = y_base + sig_idx * bracket_step
    ax.plot([pos_a, pos_a, pos_b, pos_b],
            [y_top, y_top + foot_length, y_top + foot_length, y_top],
            color="black", linewidth=1.0)
    ax.text((pos_a + pos_b) / 2, y_top + foot_length * 1.3,
            f"{pval_to_stars(row.padj)}\np={row.padj:.3f}",
            ha="center", va="bottom", fontsize=8)
    sig_idx += 1

ax.set_xticks(positions)
ax.set_xticklabels(tissue_order, fontsize=11)
ax.set_ylabel("GDF15 expression (mean per patient)", fontsize=11)
ax.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.savefig(os.path.join(output_dir, "GDF15_by_tissue_type_epi_normal_split.pdf"), dpi=300, bbox_inches="tight")
plt.show()